In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.preprocessing import LabelEncoder, StandardScaler  # ⭐StandardScalerを追加
from sklearn.linear_model import Ridge  # ⭐Ridge回帰を追加
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

class Paths:
    p = "/Users/shirokoshikentaro/Desktop/Python3/house-prices-advanced-regression-techniques"
    train = p + "/train.csv"
    test = p + "/test.csv"
    sample = p + "/sample_submission.csv"

# ===== ステップ1: データ読み込み =====
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")

# ===== ステップ2: 目的変数を先に取り出す =====
y_train = np.log1p(train["SalePrice"].values)
print(f"y_train shape: {y_train.shape}")

# Id列を保存
train_id = train["Id"]
test_id = test["Id"]

# 外れ値除去
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]
y_train = np.log1p(train["SalePrice"].values)
train_id = train["Id"]

# ===== ステップ3: 新規特徴量作成 =====
for df in [train, test]:
    # 既存の特徴量
    df["QualGrLiv"] = df["OverallQual"] * df["GrLivArea"]
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["QualTotalSF"] = df["OverallQual"] * df["TotalSF"]
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalBath"] = (df["FullBath"] + 0.5*df["HalfBath"].fillna(0) + 
                       df["BsmtFullBath"].fillna(0) + 0.5*df["BsmtHalfBath"].fillna(0))
    df["AreaPerRoom"] = df["GrLivArea"] / df["TotRmsAbvGrd"].replace(0, 1)
    df["GarageScore"] = df["GarageCars"] * df["GarageArea"]

    # log変換した特徴量
    df["Log_GrLivArea"] = np.log1p(df["GrLivArea"])
    df["Log_LotArea"] = np.log1p(df["LotArea"])
    df["Log_TotalSF"] = np.log1p(df["TotalSF"])
    
    # 新規特徴量
    df["QualGrLivLog"] = df["OverallQual"] * df["Log_GrLivArea"]
    df["TotalPorchSF"] = (df["OpenPorchSF"] + df["EnclosedPorch"] + 
                          df["3SsnPorch"].fillna(0) + df["ScreenPorch"].fillna(0))
    df["IsNew"] = (df["Age"] <= 5).astype(int)
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
    df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["BsmtScore"] = df["TotalBsmtSF"] * df["HasBsmt"]

# ===== ステップ4: Id, SalePriceを削除 =====
train = train.drop(["Id", "SalePrice"], axis=1)
test = test.drop(["Id"], axis=1)

# ===== ステップ5: 数値とカテゴリを自動判定 =====
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()

print(f"数値特徴量: {len(num_cols)}個")
print(f"カテゴリ特徴量: {len(cat_cols)}個")

# ===== ステップ6: 欠損値処理 =====
print("\n欠損値処理中...")
for col in num_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

for col in cat_cols:
    train[col] = train[col].fillna("None")
    test[col] = test[col].fillna("None")

# ===== ステップ7: One-Hotエンコーディング =====
print("One-Hotエンコーディング中...")

train['is_train'] = 1
test['is_train'] = 0
combined = pd.concat([train, test], axis=0, ignore_index=True)

combined_encoded = pd.get_dummies(
    combined, 
    columns=cat_cols,
    drop_first=True,
    dtype=int
)

train_encoded = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
test_encoded = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

all_features = list(train_encoded.columns)
print(f"One-Hot後の総特徴量: {len(all_features)}個")

# ===== ステップ8: X_train, X_test作成 =====
X_train = train_encoded.values
X_test = test_encoded.values

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")

# ===== ステップ9: CV =====
params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 16,
    "learning_rate": 0.03,
    "n_estimators": 15000,
    "min_child_samples": 30,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 0.5,
    "random_state": 123,
    "verbose": -1,
}

cv = KFold(n_splits=5, shuffle=True, random_state=123)
metrics = []

print("\n===== Cross Validation =====")
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    metrics.append(rmse)
    print(f"Fold {fold}: {rmse:.5f}")

print(f"\n[CV] {np.mean(metrics):.5f}±{np.std(metrics):.5f}")

# ===== ステップ10: LightGBM予測 =====
print("\n===== LightGBM予測（5シード） =====")
lgb_predictions = []
seeds_lgb = [123, 456, 789, 42, 2024]

for seed in seeds_lgb:
    print(f"LightGBM Seed {seed} で学習中...")
    params['random_state'] = seed
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train, 
              eval_set=[(X_train, y_train)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    y_pred_log_seed = model.predict(X_test)
    lgb_predictions.append(y_pred_log_seed)

lgb_pred_mean = np.mean(lgb_predictions, axis=0)
print("✅ LightGBM予測完了")

# ===== XGBoost予測 =====
print("\n===== XGBoost予測（3シード） =====")
xgb_params = {
    'max_depth': 3,
    'learning_rate': 0.03,
    'n_estimators': 10000,
    'min_child_weight': 5,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.5,
    'reg_lambda': 0.5,
    'random_state': 123,
    'early_stopping_rounds': 100,
}

xgb_predictions = []
seeds_xgb = [123, 456, 789]

for seed in seeds_xgb:
    print(f"XGBoost Seed {seed} で学習中...")
    xgb_params['random_state'] = seed
    
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, 
              eval_set=[(X_train, y_train)],
              verbose=False)
    
    xgb_predictions.append(model.predict(X_test))

xgb_pred_mean = np.mean(xgb_predictions, axis=0)
print("✅ XGBoost予測完了")

# ===== Ridge回帰 =====  ⭐ここに追加
print("\n===== Ridge回帰 =====")

# データを標準化（Ridge回帰用）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge_predictions = []
for alpha in [10, 15, 20]:
    print(f"Ridge alpha={alpha}")
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    ridge_predictions.append(model.predict(X_test_scaled))

ridge_pred_mean = np.mean(ridge_predictions, axis=0)
print("✅ Ridge予測完了")

# ===== 3モデルのアンサンブル =====
print("\n===== アンサンブル（LightGBM + XGBoost + Ridge） =====")
y_pred_log = 0.5 * lgb_pred_mean + 0.3 * xgb_pred_mean + 0.2 * ridge_pred_mean
y_pred = np.expm1(y_pred_log)

# ===== ステップ11: 提出ファイル =====
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": y_pred
})

submission.to_csv("submission_3models_ensemble.csv", index=False)
print(f"\n✅ 保存完了！平均予測価格: ${y_pred.mean():,.0f}")


train shape: (1460, 81)
test shape: (1459, 80)
y_train shape: (1460,)
数値特徴量: 53個
カテゴリ特徴量: 43個

欠損値処理中...
One-Hotエンコーディング中...
One-Hot後の総特徴量: 283個

X_train shape: (1458, 283)
X_test shape: (1459, 283)
y_train shape: (1458,)

===== Cross Validation =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[427]	valid_0's rmse: 0.111947
Fold 0: 0.11195
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[344]	valid_0's rmse: 0.131105
Fold 1: 0.13111
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[643]	valid_0's rmse: 0.138885
Fold 2: 0.13889
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[523]	valid_0's rmse: 0.130735
Fold 3: 0.13074
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[350]	valid_0's rmse: 0.111815
Fold 4: 0.11181

[CV] 0.12490±0.01102

=====